In [1]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
import gsw


In [ ]:
dir_name = "/home/x_titmo/work/runs_output/NeverWorld2/"

# Averages 
ds_Clim_bis = xr.open_mfdataset(dir_name + "/Clim_bis/nw2_dino.0000.averages.nc", decode_times=False)
ds_Mix_Max  = xr.open_mfdataset(dir_name + "/Mix_Max/nw2_dino.0000.averages.nc", decode_times=False)
ds_No_SO    = xr.open_mfdataset(dir_name + "/No_SO/nw2_dino.0000.averages.nc", decode_times=False)

datasets_avg = {"Clim": ds_Clim_bis, "Mix_Max" : ds_Mix_Max, "No_SO" : ds_No_SO}

# Snapshots
ds_Clim_bis_snap = xr.open_mfdataset(dir_name + "/Clim_bis/nw2_dino.0000.snapshot.nc", decode_times=False)
ds_Mix_Max_snap  = xr.open_mfdataset(dir_name + "/Mix_Max/nw2_dino.0000.snapshot.nc", decode_times=False)
ds_No_SO_snap    = xr.open_mfdataset(dir_name + "/No_SO/nw2_dino.0000.snapshot.nc", decode_times=False)

datasets_snap = {"Clim": ds_Clim_bis_snap, "Mix_Max" : ds_Mix_Max_snap,  "No_SO" : ds_No_SO_snap}

# Overturning
ds_Clim_bis_ovr = xr.open_mfdataset(dir_name + "/Clim_bis/nw2_dino.0000.overturning.nc", decode_times=False)
ds_Mix_Max_ovr  = xr.open_mfdataset(dir_name + "/Mix_Max/nw2_dino.0000.overturning.nc", decode_times=False)
ds_No_SO_ovr    = xr.open_mfdataset(dir_name + "/No_SO/nw2_dino.0000.overturning.nc", decode_times=False)

datasets_ovr = {"Clim": ds_Clim_bis_ovr, "Mix_Max" : ds_Mix_Max_ovr,  "No_SO" : ds_No_SO_ovr}

ds_snap = ds_Clim_bis_snap

style = {
    "Clim": {"color": "tab:orange", "linestyle": "solid"},
    "No_SO":      {"color": "tab:red", "linestyle": "solid"},
    "Mix_Max":  {"color": "tab:purple", "linestyle": "solid"},
    }

keys_to_plot = ["Mix_Max", "No_SO", "Clim"]

In [4]:
for key in keys_to_plot :
    datasets_avg[key]  = datasets_avg[key].assign_coords(years=("Time", (datasets_avg[key].Time / 365).data))
    datasets_ovr[key]  = datasets_ovr[key].assign_coords(years=("Time", (datasets_ovr[key].Time / 365).data))
    datasets_snap[key] = datasets_snap[key].assign_coords(years=("Time", (datasets_snap[key].Time / 365).data))

land_mask = (ds_snap.zt.to_numpy()[:,np.newaxis, np.newaxis] <= -ds_snap.bathymetry.to_numpy()[np.newaxis,...]) | (ds_snap.salt.isel(Time=0).drop_vars("Time") == 0.0)

weight_xyz_t = (1-land_mask) * ds_snap.area_t * ds_snap.dzt
weight_xyz_t = weight_xyz_t.fillna(0) / weight_xyz_t.sum()
weights_xy_t = weight_xyz_t / weight_xyz_t.sum('xt').sum('yt')
weights_z_t = weight_xyz_t.mean(dim=('xt','yt'))


In [5]:
dict_field_moc = {}
dict_field_eddies = {}
dict_field_rmoc = {}
dict_field_zonal_mean_gsw_prho = {}

new_layer = xr.DataArray(
    data=np.zeros_like(ds_snap.yu),
    dims=['yu'],
    coords={'yu': ds_snap.yu}
)

# Expand to 2D with the new z coordinate
new_layer = new_layer.expand_dims(dim={'zw': [-4000]})


ytp = 1500.
for key in keys_to_plot :
    ds_ovr = datasets_ovr[key]
    ds_avg = datasets_avg[key]
    
    gsw_sig2 =  gsw.sigma2(ds_avg.salt, ds_avg.temp)
    dict_field_zonal_mean_gsw_prho[key] = gsw_sig2.mean(dim='Time').weighted(weight_xyz_t).mean(dim=('xt'))

 
    dict_field_moc[key] = -(ds_ovr.vsf_iso * 1e-6).mean(dim='Time')
    dict_field_eddies[key] = -(ds_ovr.bolus_iso * 1e-6).mean(dim='Time')
    psi_sf = (dict_field_moc[key] + dict_field_eddies[key])
    psi_extended = xr.concat([new_layer, psi_sf], dim='zw')

    dict_field_rmoc[key] = psi_extended


In [6]:
def lat_depth_plot(fig, gs, mvt, label1, label2, cmax, field_sig2=False, fld_sig2=[], bot_line=True):
    
    from matplotlib.gridspec import GridSpec
    plt.rcParams.update({'font.size': 14})


    # Set up depth coordinate
    depth = -mvt['zw']
    stretch_depth = 500

    # Set up colormap and colorbar
    cmap = 'RdBu_r'
    fld = mvt
    cmin = - cmax
    
    sig2_levels = np.hstack([np.arange(30, 38, 1),np.arange(38, 40, 1)])
    
    # First the "stretched" top plot
    ax1 = fig.add_subplot(gs[0])
    p1  = ax1.contourf(mvt['yu'],depth,fld,cmap=cmap,vmin=cmin,vmax=cmax,levels=np.arange(cmin,cmax+1,1))
    p1b = ax1.contour(mvt['yu'],depth, fld, levels=np.arange(cmin-2.5,cmax,5), colors='k')
    if field_sig2:       
        p1c = ax1.contour(mvt['yu'],-fld_sig2.zt, fld_sig2, levels=sig2_levels, colors='k', linestyles='dashed', linewidths=0.8)
        ax1.clabel(p1c, inline=True, fontsize=12)
    plt.grid()

    # Handle y-axis
    ax1.invert_yaxis()
    plt.ylim([stretch_depth, 0])
    ax1.yaxis.axes.set_yticks(np.arange(stretch_depth,0,-100))

    # Remove top plot xtick label
    ax1.xaxis.axes.set_xticklabels([])

    # Now the rest ...
    ax2 = fig.add_subplot(gs[1:])
    p2  = ax2.contourf(mvt['yu'],depth, fld, cmap=cmap,vmin=cmin,vmax=cmax,levels=np.arange(cmin,cmax+1,1))
    p2b = ax2.contour(mvt['yu'],depth, fld, levels=np.arange(cmin-2.5,cmax,5), colors='k')
    if field_sig2:
        p2c = ax2.contour(mvt['yu'],-fld_sig2.zt, fld_sig2, levels=sig2_levels, colors='k', linestyles='dashed', linewidths=0.8)
        ax2.clabel(p2c, inline=True, fontsize=12)
    plt.grid()


    if bot_line :
        ax1.vlines(-65, 2500, 0, color='tab:red', linestyle='solid')
        ax2.vlines(-65, 2500, 0, color='tab:red', linestyle='solid')
        ax1.vlines(-45, 2500, 0, color='tab:red', linestyle='solid')
        ax2.vlines(-45, 2500, 0, color='tab:red', linestyle='solid')
        ax2.hlines(2500, -65, -45, color='tab:red', linestyle='solid')
  
    # Handle y-axis
    ax2.invert_yaxis()
    plt.ylim([4000, stretch_depth])
    plt.ylabel(f'Pseudo-depth [m]')
    ax2.yaxis.set_label_coords(-0.075,0.75)

    # Label  axis
    plt.xlabel('Latitude')

    # Reduce space between subplots
    fig.subplots_adjust(hspace=0.0)

    # Make a single title
    ax1.set_title(f'{label1} {label2} ',verticalalignment='top',fontsize=18)

    return p2

In [7]:
fig = plt.figure(figsize=(12,18), dpi=600)
gs = GridSpec(3, 1, figure=fig, hspace=0.25)
gs0 = gs[0].subgridspec(3, 1)
gs1 = gs[1].subgridspec(3, 1)
gs2 = gs[2].subgridspec(3, 1)
fld = dict_field_rmoc["Mix_Max"]
abs_max = np.max(np.abs([fld.min(),fld.max()]))
cmax = round(abs_max,-1)
cmax = 20

clim    = lat_depth_plot(fig, gs0, dict_field_rmoc["Clim"],r'(a) Reference case','', cmax, field_sig2=True, fld_sig2=dict_field_zonal_mean_gsw_prho["Clim"])
mix_max = lat_depth_plot(fig, gs1, dict_field_rmoc["Mix_Max"],r'(b) $\kappa_v = 1.25 cm^2.s^{-1}$','', cmax, field_sig2=True, fld_sig2=dict_field_zonal_mean_gsw_prho["Mix_Max"])
no_so   = lat_depth_plot(fig, gs2, dict_field_rmoc["No_SO"],'(c) Blocked Drake passage','', cmax, field_sig2=True, fld_sig2=dict_field_zonal_mean_gsw_prho["No_SO"], bot_line=False)

# Make the colorbar
fig.subplots_adjust(right=0.83)
cbar_ax = fig.add_axes([0.87, 0.1, 0.025, 0.8])
fig.colorbar(mix_max,cax=cbar_ax, ticks=np.arange(-cmax-2.5,cmax,5))
cbar_ax.set_ylabel(f'[Sv]')

path_to_plot = f"/home/x_titmo/work/analysis/NeverWorld2/plots/review_goc/"
#plt.savefig(path_to_plot  + f'Clim_MixMax_NoSO_gsw_iso_750y.pdf', bbox_inches='tight')
